# 06 – Character Nicknames

Esplorazione e data cleaning del dataset `character_nicknames.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `character_mal_id` | Chiave esterna che fa riferimento alla chiave primaria di `characters.csv` |
| `nickname` | Soprannome del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_nicknames = pd.read_csv('../datasets/character_nicknames.csv')
print(f'Shape: {df_nicknames.shape}')
print()
df_nicknames.info()
df_nicknames.head()

Shape: (37080, 2)

<class 'pandas.DataFrame'>
RangeIndex: 37080 entries, 0 to 37079
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   character_mal_id  37080 non-null  int64
 1   nickname          37064 non-null  str  
dtypes: int64(1), str(1)
memory usage: 579.5 KB


,character_mal_id,nickname
0,280205,Hikaruko
1,280129,Hinacchi
2,280127,Bertha Willis
3,280066,Jimmy
4,280059,Full Body Red Square


**Osservazioni iniziali:**
- Il dataset contiene **37.080 righe** e **2 colonne**.
- `character_mal_id` è completo: **nessun valore nullo**
- `nickname` presenta **16 valori nulli** (37.064 non-null su 37.080), da gestire in fase di cleaning.
- I tipi di dati sono adeguati: `int64` e `str`

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_nicknames)

mask_dup = df_nicknames.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_nicknames[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_nicknames.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_nicknames):,}')

Righe totali coinvolte in duplicazioni : 168
  → prime occorrenze mantenute         : 66
  → occorrenze extra rimosse           : 102

Righe prima della rimozione : 37,080
Righe dopo la rimozione     : 36,978


**Osservazioni**

**168 righe** (0.45% del totale) risultano coinvolte in duplicazioni esatte su entrambe le colonne. Di queste, **66** sono prime occorrenze mantenute e **102** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 37.080 a **36.978 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `characters.csv`. Per una chiave esterna le statistiche descrittive non hanno significato interpretativo.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `characters_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

I valori duplicati in questo caso sono attesi in quanto lo stesso personaggio può avere più di un alias. 

In [3]:
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv')

mask_orphan = check_fk(
    df_nicknames['character_mal_id'],
    df_characters['character_mal_id'],
    child_df=df_nicknames
)

print(f'Null in character_mal_id  : {df_nicknames["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_nicknames["character_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        character_mal_id
  Colonna PK  (tabella padre)         character_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       36,978
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            36,978  (100.00%)
  Valori unici nella PK padre         209,506

  ✓  Righe con FK valida              36,978  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in character_mal_id  : 0
Duplicati in character_mal_id (attesi) : 6,696


**Osservazioni:**
- **Nessun valore nullo**: tutti i 36.978 record hanno un ID di riferimento.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia è necessaria.

### 2.2 `nickname`

È l'unica colonna con contenuto semantico del dataset: contiene il soprannome del personaggio in forma testuale libera.

A differenza di `character_mal_id`, qui ci interessa capire la distribuzione delle lunghezze, la presenza di valori nulli, duplicati, caratteri anomali, etc. Per questo utilizziamo `analyze`. 

I duplicati sono **attesi** in quanto soprannomi generici possono comparire per personaggi diversi e non costituiscono un problema.

In [4]:
df_nicknames['nickname'] = df_nicknames['nickname'].str.strip()
analyze(df_nicknames['nickname'])


  Nome serie                     nickname
  dtype                          str
  Dimensione                     36,978
  Range indice                   0 … 37079
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   36,978
  Valori non nulli               36,960  (99.95%)
  Null / NaN                     18  (0.05%)
    di cui stringhe vuote        2  (convertite a NULL)
  Valori unici                   28,891  (78.13%)
  Valori duplicati               8,069  (21.82%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  87  caratteri
  Lunghezza media                9.83  caratteri
  Lunghezza mediana              8.0000  caratteri
  Dev. standard lunghezza        5.78
  IQR lunghezza                  7.0000

Valori di lung

**Osservazioni:**
- Ci sono **18 valori nulli** da cui 2 vuoti che sono righe senza nickname prive di significato e sono da rimuovere.
- Ci sono **28.928 valori unici** (78.23%) e i nickname duplicati (21.77%) sono attesi quindi non c'è nessuna anomalia.
- Lunghezza variabile da **1 carattere** a **87 caratteri**. Media: 9.84 caratteri. Stampiamo i 10 più lunghi e 10 più corti. 
- Il 57.4% dei nickname ha lunghezza tra 1 e 9 caratteri, confermando la prevalenza di soprannomi brevi.
- **54 nickname puramente numerici** che stampiamo per verificare se ci sono delle anomalie. 
- **12.3% contiene caratteri speciali**. Analizziamo più in profondità i seguenti casi:
  - **4 nickname contengono `@`**: da verificare (possibili typo).
  - **nickname contenenti `;`**: probabili separatori tra alias multipli, da spezzare su righe distinte.

In [5]:
lunghezze = df_nicknames['nickname'].str.len()

top10_lunghi = df_nicknames.loc[lunghezze.nlargest(10).index].assign(caratteri=lunghezze)
top10_corti  = df_nicknames.loc[lunghezze.nsmallest(10).index].assign(caratteri=lunghezze)

print('10 nickname più lunghi:')
print(top10_lunghi[['character_mal_id', 'nickname', 'caratteri']].to_string(index=False))
print()
print('10 nickname più corti:')
print(top10_corti[['character_mal_id', 'nickname', 'caratteri']].to_string(index=False))
print()

mask_num = df_nicknames['nickname'].str.fullmatch(r'\d+', na=False)
print(f'Nickname solo numerici: {mask_num.sum()}')
print()
print(df_nicknames[mask_num][['character_mal_id', 'nickname']].to_string(index=False))

10 nickname più lunghi:
 character_mal_id                                                                                nickname  caratteri
           213389 Laplus Dear Highest Death 13 Dyna Art of Impact Sign Emperor Lord of the Darknesss-sama       87.0
           108303     Crimson Snake; Chi Lian; Chiliannv; Crimson Lotus; Princess Honglian; Princess Lian       83.0
           230023                      Kotarou Hill Mountain Williams Harrison Jagasatake James Shiroyama       66.0
           146509                      Black Shadow; Darkness; The Demon Duelist; The Supreme King Dragon       66.0
            47693                             Woman who Wished the Fallen wings Good Tidings and Farewell       59.0
           240069                              Eight-Handled Sword Divergent Sila Divine General Mahoraga       58.0
           131636                              The Scar Faced Dark Messiah; The Rule-Smashing Pun Machine       58.0
            27883                       

##### Nickname con `@`

Quattro nickname contengono il carattere `@`, non atteso in un soprannome testuale. Potrebbe trattarsi di un typo. Visualizziamo le righe per decidere la correzione.

In [6]:
mask_at = df_nicknames['nickname'].str.contains('@', regex=False, na=False)
print(f'Righe con @ nel nickname: {mask_at.sum()}')
df_nicknames[mask_at]

Righe con @ nel nickname: 4


,character_mal_id,nickname
14077,150363,@NyanNyan
14078,150363,@Meow-Meow
14687,144978,S@ik1 KAzU
30570,18138,Riko @ Androids


Si nota che sono tutti da correggere. Ci sono 3 casi dove si tratta di caratteri extra da rimuovere, e un caso dove il carattere `@` è una sostituzione della lettera `a`. Correggiamo gli errori di battitura. 

In [7]:
corrections = {
    '@NyanNyan':       'NyanNyan',
    '@Meow-Meow':      'Meow-Meow',
    'S@ik1 KAzU':      'Saik1 KAzU',
    'Riko @ Androids': 'Riko Androids',
}

df_nicknames['nickname'] = df_nicknames['nickname'].replace(corrections)

remaining = df_nicknames['nickname'].str.contains('@', regex=False, na=False).sum()
print(f'Nickname con @ dopo la correzione: {remaining}')

Nickname con @ dopo la correzione: 0


##### Nickname con `;`

Il carattere `;` compare in alcune righe come separatore di alias multipli nello stesso campo. Queste righe andrebbero spezzate. Verifichiamo quante sono e come si presentano.

In [8]:
mask_semi = df_nicknames['nickname'].str.contains(';', regex=False, na=False)
print(f'Righe con ; nel nickname: {mask_semi.sum()}')
df_nicknames[mask_semi].head(10)

n_prima = len(df_nicknames)

df_nicknames['nickname'] = df_nicknames['nickname'].str.split(';')
df_nicknames = df_nicknames.explode('nickname').reset_index(drop=True)
df_nicknames['nickname'] = df_nicknames['nickname'].str.strip()

# Rimuoviamo eventuali stringhe vuote generate da ; finali
df_nicknames = df_nicknames[df_nicknames['nickname'] != ''].reset_index(drop=True)

n_dopo = len(df_nicknames)
print(f'Righe prima della separazione : {n_prima:,}')
print(f'Righe dopo la separazione     : {n_dopo:,}')
print(f'Nuove righe generate           : {n_dopo - n_prima:,}')

Righe con ; nel nickname: 65
Righe prima della separazione : 36,978
Righe dopo la separazione     : 37,057
Nuove righe generate           : 79


### 2.3 Chiave primaria `(character_mal_id, nickname)`

Dopo tutte le operazioni di pulizia, verifichiamo che la coppia `(character_mal_id, nickname)` sia univoca: un personaggio non dovrebbe avere lo stesso soprannome registrato due volte.

Infine rimuoviamo le 16 righe con `nickname` nullo rimaste dal dataset originale.

In [9]:
n_pk_dup = df_nicknames.duplicated(subset=['character_mal_id', 'nickname']).sum()
print(f'Duplicati su (character_mal_id, nickname): {n_pk_dup}')
if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_nicknames.drop_duplicates(subset=['character_mal_id', 'nickname'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_nicknames):,}')
else:
    print('→ Chiave primaria già univoca.')

print()

n_null_nickname = df_nicknames['nickname'].isna().sum()
print(f'Null in nickname prima della pulizia: {n_null_nickname}')
df_nicknames.dropna(subset=['nickname'], inplace=True)
print(f'Null in nickname dopo la pulizia    : {df_nicknames["nickname"].isna().sum()}')
print(f'Righe dopo pulizia nickname         : {len(df_nicknames):,}')

Duplicati su (character_mal_id, nickname): 0
→ Chiave primaria già univoca.

Null in nickname prima della pulizia: 16
Null in nickname dopo la pulizia    : 0
Righe dopo pulizia nickname         : 37,041


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [10]:
righe_aggiunte = n_dopo - n_prima
righe_rimosse  = n_originale - len(df_nicknames) + righe_aggiunte

print('Riepilogo Dataset Pulito')
print(f'Righe originali            : {n_originale:>10,}')
print(f'Righe rimosse              : {righe_rimosse:>10,}')
print(f'Righe aggiunte (split ";") : {righe_aggiunte:>10,}')
print(f'Righe dopo cleaning        : {len(df_nicknames):>10,}')

print()
df_nicknames.to_csv('../datasets_cleaned/character_nicknames_clean.csv', index=False)
print('Salvato: datasets_cleaned/character_nicknames_clean.csv')

Riepilogo Dataset Pulito
Righe originali            :     37,080
Righe rimosse              :        118
Righe aggiunte (split ";") :         79
Righe dopo cleaning        :     37,041

Salvato: datasets_cleaned/character_nicknames_clean.csv
